# 18 · I device mandano la telemetria
*Blocco 5 · Dati in Fabric. Passo di esecuzione.*

Due device nuovi, uno per Hub, si registrano a DPS (il certificato lo emette **ADR**) e mandano ciascuno alcuni messaggi **con quel certificato**. L'Hub li accetta perché si fida della CA, non del singolo device, e li inoltra alla Eventstream.

**Prima:** i notebook 01, 02a-02c, 17a e 17b.

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

In [ ]:
$device = @(                           # i device da lanciare: il gruppo decide a quale Hub vanno
    @{ Id = 'robot-03'; Group = 'robots' },
    @{ Id = 'truck-03'; Group = 'trucks' }
)
$messaggi = 8                          # messaggi per device. MODIFICA = 0 per non fermarsi mai (si chiude la finestra o Ctrl+C)
$intervallo = 3                        # secondi tra due messaggi
$finestre = $false                     # MODIFICA = $true per una finestra visibile per ogni device

## I device partono
Ogni device è un processo `_run-device.ps1` con la sua cartella di stato. Parte da zero: chiave, richiesta di certificato, registrazione, poi i messaggi.

In [ ]:
$procs = foreach ($d in $device) {
    $state = Get-ScenarioDir -Config $cfg -Notebook '18-device-telemetria' -DeviceId $d.Id
    $boot = Join-Path $cfg.Out "bootstrap-ca-$($d.Group)"
    Start-DemoDevice -RunScript $runScript -DeviceId $d.Id -StateDir $state -BootstrapCaDir $boot -DpsName $cfg.DpsName -ResourceGroup $cfg.Rg `
        -IntervalSeconds $intervallo -Count $messaggi -Hidden:(-not $finestre)
}
"Partiti: " + (($procs | ForEach-Object { "pid $($_.Id)" }) -join ', ')

## Cosa dicono
Aspettiamo che finiscano e leggiamo le ultime righe di ciascuno: Hub assegnato, certificato, messaggi inviati.

In [ ]:
if (-not $finestre -and $messaggi -gt 0) {
    $procs | Wait-Process -Timeout 300 -ErrorAction SilentlyContinue
    foreach ($d in $device) {
        "=== $($d.Id)"
        Get-Content (Join-Path (Get-ScenarioDir -Config $cfg -Notebook '18-device-telemetria' -DeviceId $d.Id) 'stdout.log') -ErrorAction SilentlyContinue | Select-Object -Last 5
        $err = Join-Path (Get-ScenarioDir -Config $cfg -Notebook '18-device-telemetria' -DeviceId $d.Id) 'stderr.log'
        if ((Test-Path $err) -and (Get-Item $err).Length -gt 0) { "ERRORE del device:"; Get-Content $err }      # senza questo un device che non parte sembra solo silenzioso
    }
}

Prossimo passo: **19**, si controlla dove sono arrivati i messaggi.